In [18]:
from ssb import *
from utils.universal_data_loader import *

# Example Feature Groupings Setup
# business_groups = {
# "delinquency_vars":["dpd_avg_3m", "dpd_avg_6m", "dpd_avg_12m", "max_dpd_12m"],            
# "transaction_vars":["txn_count_avg_3m", "txn_count_avg_6m", "txn_count_avg_12m"],
# "spend_vars":["spend_avg_3m", "spend_avg_6m", "spend_avg_12m"],
# "repayment_vars":["payment_ratio_avg_3m", "payment_ratio_avg_6m", "payment_ratio_avg_12m"],
# "card_utilization_vars":["utilization_avg_3m", "utilization_avg_6m", "utilization_avg_12m", "utilization_max_12m"],
# }

# 2. Define your dynamically scaled optimization grid
my_search_grid = {
    "min_sample_size": [10000, 5000],
    "min_lift": [2.0,1.5]
}


# Initialize the builder
builder = StrategicSegmentBuilder(
    target="default_flag",
    n_jobs=-1,
    min_sample_size=5000,
    min_lift=1.5,
    top_n_vars=8,
    max_segments=10,
    enable_diversity=True,
    max_feature_reuse=99,
    enable_1way=True,  # Include 1-way rules from final output
    enable_2way=True,  # Exclude 2-way rules from final output
    enable_3way=True,  # Exclude 3-way rules from final output
    feature_groups=None, #business_groups,
    ignore_features=None,
    param_grid=my_search_grid
)


data = UniversalDataLoader(file_path=r"/workspaces/Strategic_Segment_Builder/Datasets/all_categorical_credit_default_50000.csv").load()

segments_df = builder.extract_segments(data)
final_eval = builder.evaluate_final_coverage(data)

2026-07-04 06:29:58,010 | INFO     | [universal_data_loader.py:75] | Detecting local file format handler for extension: '.csv'...
2026-07-04 06:29:58,099 | INFO     | [ssb.py:338] | Dynamic Grid Search Enabled: 4 total configurations.
2026-07-04 06:29:58,100 | INFO     | [ssb.py:359] | Iteration 1 | Remaining Volume: 50,000 | Base Rate: 14.77%
2026-07-04 06:29:59,998 | INFO     | [ssb.py:459] | Feature Usage Tracker Update -> 'utilization_band' used count = 1
2026-07-04 06:29:59,999 | INFO     | [ssb.py:474] | Segment 1 Captured (Size Floor: 5000 | Lift Floor: 1.5): utilization_band IN ('VeryHigh')
2026-07-04 06:30:00,001 | INFO     | [ssb.py:359] | Iteration 2 | Remaining Volume: 42,401 | Base Rate: 12.58%
2026-07-04 06:30:01,529 | INFO     | [ssb.py:459] | Feature Usage Tracker Update -> 'risk_segment' used count = 1
2026-07-04 06:30:01,530 | INFO     | [ssb.py:474] | Segment 2 Captured (Size Floor: 5000 | Lift Floor: 1.5): risk_segment IN ('SubPrime')
2026-07-04 06:30:01,533 | INFO 

In [15]:
from prettytable import PrettyTable
import pandas as pd
table = PrettyTable()
table.field_names = list(pd.DataFrame(final_eval).columns)
for _, row in pd.DataFrame(final_eval).iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+--------------+--------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate | capture_rate |        lift        |
+---------+-------------+---------------+--------------------+--------------------+--------------+--------------------+
|   0.0   |   33776.0   |     3532.0    | 10.457129322595927 |       14.768       |    67.552    | 0.7080938057012409 |
|   1.0   |    7599.0   |     2052.0    | 27.003553099091985 |       14.768       |    15.198    | 1.8285179509135958 |
|   2.0   |    8625.0   |     1800.0    | 20.869565217391305 |       14.768       |    17.25     | 1.4131612416976778 |
+---------+-------------+---------------+--------------------+--------------------+--------------+--------------------+


In [16]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   utilization_band=<ArrowStringArray>
['VeryHigh']
Length: 1, dtype: str
SQL Filter: utilization_band IN ('VeryHigh')
--------------------------------------------------
Segment ID: 2
Raw Rule:   risk_segment=<ArrowStringArray>
['SubPrime']
Length: 1, dtype: str
SQL Filter: risk_segment IN ('SubPrime')
--------------------------------------------------


In [13]:
pd.DataFrame(final_eval)

,segment,total_count,target_events,response_rate,base_response_rate,capture_rate,lift
0,0,33776,3532.0,10.457129,14.768,67.552,0.708094
1,1,7599,2052.0,27.003553,14.768,15.198,1.828518
2,2,8625,1800.0,20.869565,14.768,17.250,1.413161


In [19]:
# Preparing the dataset for scoring and decile banding.
conn = duckdb.connect()
predicted = conn.register("predicted", data)
predicted = conn.query("""
    SELECT *, 
    CASE WHEN utilization_band IN ('VeryHigh') THEN 1 ELSE 0 END AS seg_1,
    CASE WHEN risk_segment IN ('SubPrime')
 THEN 1 ELSE 0 END AS seg_2,
    FROM predicted
""").df()
conn.close()

In [21]:
scorer = StrategicSegmentScore(
    target_col="default_flag",
    primary_key="index",
    segment_cols=["seg_1", "seg_2"],
)

In [22]:
scorer.calculate_and_export_weights(predicted, export_path="scorecard_model_test1.json")

2026-07-04 06:31:17,192 | INFO     | [ssb.py:546] | Initializing DuckDB scorecard engine...
2026-07-04 06:31:17,340 | INFO     | [ssb.py:581] | Computing harmonic scorecard weights...
2026-07-04 06:31:17,341 | INFO     | [ssb.py:618] | Scoring training dataset via NumPy Linear Algebra engine...
2026-07-04 06:31:17,344 | INFO     | [ssb.py:635] | Dataset Zero-Inflation Rate: 85.23%
2026-07-04 06:31:17,345 | INFO     | [ssb.py:638] | High Zero-Inflation (>=80%). Isolating Active Population...
2026-07-04 06:31:17,345 | INFO     | [ssb.py:650] | Calibrating deciles across 16,224 target customers...


{'model_metadata': {'total_training_population': 50000,
  'active_scored_population': 16224,
  'active_population_pct': 32.45,
  'baseline_event_rate': 0.1477},
 'segment_weights': {'seg_1': {'weight': 50,
   'lift': 1.8285,
   'response_rate': 0.27,
   'capture_rate': 0.2779},
  'seg_2': {'weight': 45,
   'lift': 1.6284,
   'response_rate': 0.2405,
   'capture_rate': 0.3291}},
 'decile_min_thresholds': {'1': 50,
  '2': 50,
  '3': 50,
  '4': 50,
  '5': 45,
  '6': 45,
  '7': 45,
  '8': 45,
  '9': 45,
  '10': 45}}

In [23]:
scorecard_json_path = "/workspaces/Strategic_Segment_Builder/Scores/scorecard_model_test1.json"
logging.info(f"Loading scorecard model artifact from {scorecard_json_path}...")
with open(scorecard_json_path, "r") as json_file:
    model_artifact = json.load(json_file)

2026-07-04 06:31:55,227 | INFO     | [1327506462.py:2] | Loading scorecard model artifact from /workspaces/Strategic_Segment_Builder/Scores/scorecard_model_test1.json...


In [24]:
model_artifact.get("decile_min_thresholds")

{'1': 50,
 '2': 50,
 '3': 50,
 '4': 50,
 '5': 45,
 '6': 45,
 '7': 45,
 '8': 45,
 '9': 45,
 '10': 45}

In [25]:
for key, value in model_artifact.get("segment_weights").items():
    print(f"Segment: {key} | Weight: {value['weight']}")

Segment: seg_1 | Weight: 50
Segment: seg_2 | Weight: 45


In [26]:
model_artifact.get("decile_min_thresholds")

{'1': 50,
 '2': 50,
 '3': 50,
 '4': 50,
 '5': 45,
 '6': 45,
 '7': 45,
 '8': 45,
 '9': 45,
 '10': 45}

In [31]:
conn = duckdb.connect()
scored = conn.register("scored", predicted)
scored = conn.query("""
WITH CTE AS (
    SELECT *, 
    CASE WHEN seg_1 = 1 THEN 50 ELSE 0 END AS seg_1_weighted,
    CASE WHEN seg_2 = 1 THEN 45 ELSE 0 END AS seg_2_weighted,
    FROM scored),
    CTE2 AS (
    SELECT *, (seg_1_weighted + seg_2_weighted) AS total_weight
                     FROM CTE)
SELECT *, CASE WHEN total_weight >=50 THEN 1
                    WHEN total_weight >= 50 THEN 2
                    WHEN total_weight >= 50 THEN 3
                    WHEN total_weight >= 50 THEN 4
                    WHEN total_weight >= 45 THEN 5
                    WHEN total_weight >= 45 THEN 6
                    WHEN total_weight >= 45 THEN 7
                    WHEN total_weight >= 45 THEN 8
                    WHEN total_weight >= 45 THEN 9
                    WHEN total_weight >= 45 THEN 10
                    ELSE 0 END AS decile_band
                    
                     FROM CTE2
""").to_df()
conn.close()

In [32]:
conn = duckdb.connect()
scored = conn.register("scored", scored)
scored = conn.query("""SELECT decile_band, 
                    COUNT(*) AS count, 
                    SUM(default_flag) AS events, 
                    (SUM(default_flag)*100.0/COUNT(*)) AS response_rate
FROM scored
GROUP BY decile_band
ORDER BY decile_band
""").to_df()
conn.close()

In [33]:
scored

,decile_band,count,events,response_rate
0,0,33776,3532.0,10.457129
1,1,7599,2052.0,27.003553
2,5,8625,1800.0,20.869565
